In [1]:
from collections import deque
import time


def bfs(graph, start, goal):
    visited = set()
    queue = deque([(start, [])])

    while queue:
        node, path = queue.popleft()
        if node == goal:
            return path + [node]
        if node not in visited:
            visited.add(node)
            for neighbor in graph.get(node, []):
                if neighbor not in visited:
                    queue.append((neighbor, path + [node]))
    return None


# Graph 1 (Fig. 1 Simple Graph)
graph_1 = {
    "A": ["B", "C"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B"],
    "E": ["B", "F"],
    "F": ["C", "E"],
}

# Graph 2 (Fig. 2 Extended Graph with G and H)
graph_2 = {
    "A": ["B", "C"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B", "G"],
    "E": ["B", "F"],
    "F": ["C", "E"],
    "G": ["D", "H"],
    "H": ["G", "F"],
}

print("=== EXERCISE 1: BFS ON GRAPH 1 & GRAPH 2 ===")

test_cases = [("A", "E", graph_1), ("A", "F", graph_1), ("A", "H", graph_2)]

for start_n, goal_n, g in test_cases:
    t0 = time.perf_counter()
    path = bfs(g, start_n, goal_n)
    elapsed = (time.perf_counter() - t0) * 1000
    print(
        f"BFS from {start_n} to {goal_n} -> Path: {path} | Execution Time: {elapsed:.4f} ms"
    )

=== EXERCISE 1: BFS ON GRAPH 1 & GRAPH 2 ===
BFS from A to E -> Path: ['A', 'B', 'E'] | Execution Time: 0.0117 ms
BFS from A to F -> Path: ['A', 'C', 'F'] | Execution Time: 0.0088 ms
BFS from A to H -> Path: ['A', 'B', 'D', 'G', 'H'] | Execution Time: 0.0127 ms


In [2]:
import time


def dfs(graph, node, goal, visited=None):
    if visited is None:
        visited = set()
    if node == goal:
        return [node]

    visited.add(node)
    for neighbor in graph.get(node, []):
        if neighbor not in visited:
            path = dfs(graph, neighbor, goal, visited.copy())
            if path:
                return [node] + path
    return None


print("=== EXERCISE 2: DFS vs. BFS BENCHMARK (GRAPH 2) ===")

pairs = [("A", "H"), ("A", "E"), ("B", "F")]

for s, g in pairs:
    # BFS Benchmark
    t_start = time.perf_counter()
    b_path = bfs(graph_2, s, g)
    t_bfs = (time.perf_counter() - t_start) * 1000

    # DFS Benchmark
    t_start = time.perf_counter()
    d_path = dfs(graph_2, s, g)
    t_dfs = (time.perf_counter() - t_start) * 1000

    print(f"\n--- Query: {s} to {g} ---")
    print(f"BFS Path: {b_path} | Time: {t_bfs:.4f} ms (Optimal Hops)")
    print(f"DFS Path: {d_path} | Time: {t_dfs:.4f} ms")

=== EXERCISE 2: DFS vs. BFS BENCHMARK (GRAPH 2) ===

--- Query: A to H ---
BFS Path: ['A', 'B', 'D', 'G', 'H'] | Time: 0.0127 ms (Optimal Hops)
DFS Path: ['A', 'B', 'D', 'G', 'H'] | Time: 0.0090 ms

--- Query: A to E ---
BFS Path: ['A', 'B', 'E'] | Time: 0.0067 ms (Optimal Hops)
DFS Path: ['A', 'B', 'D', 'G', 'H', 'F', 'E'] | Time: 0.0090 ms

--- Query: B to F ---
BFS Path: ['B', 'E', 'F'] | Time: 0.0082 ms (Optimal Hops)
DFS Path: ['B', 'A', 'C', 'F'] | Time: 0.0037 ms


In [3]:
def dls(graph, node, goal, depth, visited=None):
    if visited is None:
        visited = set()
    if depth == 0:
        return None
    if node == goal:
        return [node]

    visited.add(node)
    for neighbor in graph.get(node, []):
        if neighbor not in visited:
            path = dls(graph, neighbor, goal, depth - 1, visited.copy())
            if path:
                return [node] + path
    return None


print("=== EXERCISE 3: DEPTH-LIMITED SEARCH (LDS) ===")

start_node = "A"
goal_node = "H"

# Testing cutoff thresholds
for limit in [2, 3, 4, 5]:
    res = dls(graph_2, start_node, goal_node, limit)
    if res:
        print(f"Depth Limit = {limit}: Success -> Path found: {res}")
    else:
        print(
            f"Depth Limit = {limit}: Cutoff Reached -> No path found (Target is deeper than limit)"
        )

=== EXERCISE 3: DEPTH-LIMITED SEARCH (LDS) ===
Depth Limit = 2: Cutoff Reached -> No path found (Target is deeper than limit)
Depth Limit = 3: Cutoff Reached -> No path found (Target is deeper than limit)
Depth Limit = 4: Cutoff Reached -> No path found (Target is deeper than limit)
Depth Limit = 5: Success -> Path found: ['A', 'B', 'D', 'G', 'H']


In [4]:
from collections import deque
import copy
import time


# 1. BFS Maze Solver (Optimal Shortest Route)
def bfs_maze(grid, start, goal):
    maze = copy.deepcopy(grid)
    rows, cols = len(maze), len(maze[0])
    visited = {start}
    parent = {}

    def is_valid(x, y):
        return (
            0 <= x < rows
            and 0 <= y < cols
            and maze[x][y] != "#"
            and (x, y) not in visited
        )

    queue = deque([start])
    found = False

    while queue:
        x, y = queue.popleft()
        if (x, y) == goal:
            found = True
            break
        moves = [(0, 1), (1, 0), (0, -1), (-1, 0)]  # right, down, left, up
        for dx, dy in moves:
            nx, ny = x + dx, y + dy
            if is_valid(nx, ny):
                queue.append((nx, ny))
                visited.add((nx, ny))
                parent[(nx, ny)] = (x, y)

    if not found:
        return maze, "No path found", 0

    path = []
    curr = goal
    while curr != start:
        path.append(curr)
        curr = parent[curr]
    path.reverse()

    for x, y in path:
        if maze[x][y] not in ("S", "G"):
            maze[x][y] = "*"
    return maze, "Path found", len(path)


# 2. DFS Maze Solver (Exhaustive Exploration)
def dfs_maze(grid, start, goal):
    maze = copy.deepcopy(grid)
    rows, cols = len(maze), len(maze[0])
    visited = set()
    path = []

    def is_valid(x, y):
        return (
            0 <= x < rows
            and 0 <= y < cols
            and maze[x][y] != "#"
            and (x, y) not in visited
        )

    def dfs_step(x, y):
        if (x, y) == goal:
            return True
        visited.add((x, y))
        path.append((x, y))

        moves = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        for dx, dy in moves:
            nx, ny = x + dx, y + dy
            if is_valid(nx, ny):
                if dfs_step(nx, ny):
                    return True
        path.pop()
        return False

    found = dfs_step(start[0], start[1])
    if found:
        for x, y in path:
            if maze[x][y] not in ("S", "G"):
                maze[x][y] = "*"
        return maze, "Path found", len(path)
    return maze, "No path found", 0


# Maze Grid from Workshop Document
base_maze = [
    ["S", ".", ".", ".", "#", ".", ".", "."],
    ["#", "#", "#", "#", "#", ".", "#", "#"],
    [".", ".", ".", ".", ".", ".", ".", "G"],
    ["#", "#", "#", "#", "#", "#", "#", "#"],
]

start_pos = (0, 0)
goal_pos = (2, 7)

print("=== EXERCISE 4: MAZE SOLVER COMPARISON ===")

# Run BFS
t0 = time.perf_counter()
b_grid, b_status, b_steps = bfs_maze(base_maze, start_pos, goal_pos)
t_bfs = (time.perf_counter() - t0) * 1000

# Run DFS
t0 = time.perf_counter()
d_grid, d_status, d_steps = dfs_maze(base_maze, start_pos, goal_pos)
t_dfs = (time.perf_counter() - t0) * 1000

print(
    f"\nBFS Result: {b_status} | Steps: {b_steps} | Time: {t_bfs:.4f} ms (Optimal)"
)
for row in b_grid:
    print(" ".join(row))

print(f"\nDFS Result: {d_status} | Steps: {d_steps} | Time: {t_dfs:.4f} ms")
for row in d_grid:
    print(" ".join(row))

=== EXERCISE 4: MAZE SOLVER COMPARISON ===

BFS Result: No path found | Steps: 0 | Time: 0.1038 ms (Optimal)
S . . . # . . .
# # # # # . # #
. . . . . . . G
# # # # # # # #

DFS Result: No path found | Steps: 0 | Time: 0.0812 ms
S . . . # . . .
# # # # # . # #
. . . . . . . G
# # # # # # # #
